# Lorenz Attractor

Chaotic dynamics, inline figures, streaming output.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def lorenz(state, sigma=10, rho=28, beta=8/3):
    x, y, z = state
    return np.array([sigma*(y-x), x*(rho-z)-y, x*y - beta*z])

# RK4 integration
dt, steps = 0.005, 12000
traj = np.zeros((steps, 3))
traj[0] = [1, 1, 1]
for i in range(1, steps):
    s = traj[i-1]
    k1 = lorenz(s)
    k2 = lorenz(s + dt/2 * k1)
    k3 = lorenz(s + dt/2 * k2)
    k4 = lorenz(s + dt * k3)
    traj[i] = s + (dt/6) * (k1 + 2*k2 + 2*k3 + k4)

print(f'integrated {steps} steps')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'projection': '3d'}, facecolor='#0d1926')
ax.scatter(traj[:, 0], traj[:, 1], traj[:, 2],
           c=np.linspace(0, 1, steps), cmap='inferno', s=0.1, alpha=0.6)
ax.set_facecolor('#0d1926')
ax.set_xlabel('X', color='white')
ax.set_ylabel('Y', color='white')
ax.set_zlabel('Z', color='white')
ax.tick_params(colors='#7ab8b8')
ax.set_title('Lorenz Attractor', color='white', fontsize=14)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), facecolor='#0d1926')
ax.scatter(traj[:, 0], traj[:, 2], c=np.linspace(0, 1, steps),
           cmap='cool', s=0.05, alpha=0.4)
ax.set_facecolor('#0d1926')
ax.set_xlabel('X', color='white')
ax.set_ylabel('Z', color='white')
ax.set_title('X-Z Phase Projection', color='white', fontsize=14)
ax.tick_params(colors='#7ab8b8')
ax.spines[:].set_color('#4a5a6a')
plt.tight_layout()

## Sensitivity to Initial Conditions

Progress bar + verbose output demo.

In [ ]:
import time, sys

n_trials = 50
lyapunov_estimates = []

for trial in range(n_trials):
    # progress bar
    pct = (trial + 1) / n_trials
    bar = '█' * int(pct * 30) + '░' * (30 - int(pct * 30))
    print(f'\r  {bar} {pct*100:5.1f}% | trial {trial+1}/{n_trials}', end='', flush=True)
    
    perturbation = 1e-9
    state_a = np.array([1.0, 1.0, 1.0]) + np.random.randn(3) * 0.01
    state_b = state_a + np.array([perturbation, 0, 0])
    
    for _ in range(2000):
        state_a += dt * lorenz(state_a)
        state_b += dt * lorenz(state_b)
    
    div = np.linalg.norm(state_b - state_a)
    lyapunov_estimates.append(np.log(div / perturbation) / (2000 * dt))
    time.sleep(0.05)

print(f'\n\nMax Lyapunov exponent: λ ≈ {np.mean(lyapunov_estimates):.3f} ± {np.std(lyapunov_estimates):.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), facecolor='#0d1926')
ax.hist(lyapunov_estimates, bins=15, color='#ff6b6b', edgecolor='#0d1926', alpha=0.8)
ax.axvline(np.mean(lyapunov_estimates), color='#7ab8b8', linestyle='--', linewidth=2,
           label=f'μ = {np.mean(lyapunov_estimates):.2f}')
ax.set_facecolor('#0d1926')
ax.set_xlabel('λ estimate', color='white')
ax.set_ylabel('count', color='white')
ax.set_title('Lyapunov Exponent Distribution', color='white', fontsize=14)
ax.tick_params(colors='#7ab8b8')
ax.spines[:].set_color('#4a5a6a')
ax.legend(facecolor='#0d1926', edgecolor='#4a5a6a', labelcolor='white')
plt.tight_layout()

In [ ]:
# Verbose output: trajectory statistics
print('=' * 60)
print(f'{"TRAJECTORY STATISTICS":^60}')
print('=' * 60)
for i, label in enumerate(['X', 'Y', 'Z']):
    col = traj[:, i]
    print(f'\n  {label}-axis:')
    print(f'    min    = {col.min():12.4f}')
    print(f'    max    = {col.max():12.4f}')
    print(f'    mean   = {col.mean():12.4f}')
    print(f'    std    = {col.std():12.4f}')
    print(f'    range  = {col.max() - col.min():12.4f}')
print('\n' + '=' * 60)
print(f'  Total path length: {np.sum(np.linalg.norm(np.diff(traj, axis=0), axis=1)):.2f}')
print(f'  Time span: {steps * dt:.1f} units')
print(f'  Mean speed: {np.mean(np.linalg.norm(np.diff(traj, axis=0), axis=1)) / dt:.2f}')
print('=' * 60)